In [0]:

%sql
USE CATALOG spg;
CREATE SCHEMA IF NOT EXISTS bronze;


In [0]:
%sql
-- Ingestion log
CREATE TABLE IF NOT EXISTS bronze.active_sources (
    read_ID BIGINT GENERATED ALWAYS AS IDENTITY,
    src_name STRING, 
    Active_from TIMESTAMP,
    Active_to TIMESTAMP
)
USING DELTA;

In [0]:
URL = f"http://{serverip}:13007/ticket/day.json/20180401?appid={appid}[&arguments]/"

In [0]:
from datetime import datetime
import os
# - - - - - - Paths - - - - - - - -
source_path = "abfss://stureplansgruppen@casestoragevimanvestberg.dfs.core.windows.net/files/" # Path to file folder
raw_base = "abfss://stureplansgruppen@spgt.dfs.core.windows.net/raw/"
catalog = 'spgt'
acive_sources_log = f"{catalog}.bronze.active_sources"

sources = {
    'Sperling' : 'path_unknown', 
    'Umarell' : 'path_unknown'
}

src = dbutils.fs.ls(source_path)

# - - - - Adds folder path to sources
for s in src:
    folder_name = s.name.split('/')[0]
    path = f"{source_path}{folder_name}"
    sources[folder_name] = path
    query = f"""
        SELECT distinct(src_name)
        FROM {acive_sources_log}
    """
    result_query = spark.sql(query)
    active_sources = [row.src_name for row in result_query.collect()]

    if folder_name in active_sources:
        continue
    
    else:
        spark.sql(f"""
            INSERT INTO {acive_sources_log}
            (src_name, Active_from, Active_to)
            VALUES (
                '{folder_name}',
                current_timestamp(),
                NULL
            )
        """)


# writing file to raw_base/YYYY/MM/DD/src/

dt = datetime.now()
year, month, day = dt.year, dt.month, dt.day
for src in sources:
    path = sources[src]
    files_in_src = dbutils.fs.ls(path)
    for f in files_in_src:
        file_name = f.name
        file_path = f"{source_path}{src}/{file_name}"
        dest_path = f"{raw_base}{year}/{month:02d}/{day:02d}/{src}/{file_name}"

        dbutils.fs.cp(file_path, dest_path) # copy file to location in raw



# dest_path = f"{raw_base}/{year}/{month:02d}/{day:02d}/{file_name}"


    
